In [2]:
import numpy as np
import scanpy as sc
import pandas as pd
from anndata import AnnData
from scipy.spatial.distance import cdist
import matplotlib
print(matplotlib.get_backend())
########################################################################################


# loading bdata
bdata = sc.read("/mnt/SATA/Cell2_loucura/data/input/bdata_normal_free.h5ad")
bdata.obs.rename(columns={"batch": "sample"}, inplace=True)

# correcting names
bdata.obsm["q05_cell_abundance_w_sf"].columns = bdata.uns["mod"]["factor_names"]

# normalizing rows
df = bdata.obsm["q05_cell_abundance_w_sf"]
df = df.div(df.sum(axis=1), axis=0)
bdata.obsm["q05_cell_abundance_w_sf"] = df.copy()


########################################################################################

# apenas um check que eh totalmente necessario
def spatools_check(
        adata: AnnData, 
        deconv: bool = False
    ):

    # changing uns key 
    if not deconv:
        uns_value = "spatools"
    else:
        uns_value = "deconv_spatools"

    # checking if uns value already exists
    if not uns_value in adata.uns:
        return
    
    print("Overwriting old analysis!")
    input = input("Do you want to proceed? [y or n] ").strip().lower()
    if input == "n":
        raise Exception("Operation canceled by the user.")
    elif input == "y":
        return
    else:
        raise Exception("Invalid response. Use 'y' for yes or 'n' for no.")

# pega os vizinhos com base em um limite
def measure_neighbors_filtered(
    adata: AnnData,
    freq_threshold: float = 0.2,
    distance_factor: float = 1.1,
    df_freq: pd.DataFrame = None
):
    
    # Creating a DataFrame with barcodes and spatial coordenates
    df = pd.DataFrame(
        adata.obsm["spatial"],
        index=adata.obs_names,
        columns=["x", "y"]
    )
    
    # Making sure df_freq and df are maching
    if df_freq:
        common = df.index.intersection(df_freq.index)
        df = df.loc[common]
        df_freq = df_freq.loc[common]
    else:
        try:
            df_freq = adata.obsm["q05_cell_abundance_w_sf"]
            df_freq.columns = adata.uns["mod"]["factor_names"]
        except KeyError:
            raise "'q05_cell_abundance_w_sf' does not exist inside obsm, try to give a 'df_freq' next time"
        common = df.index.intersection(df_freq.index)
        df = df.loc[common]
        df_freq = df_freq.loc[common]

    # dict to store the amount out spots per celltype analysed
    result = dict()

    # Creating dfs for all the neighbors and initiating loop
    all_neighbors_dfs = []
    for cell_type in df_freq.columns:
    
        # Filter spots with low frequence > threshold
        selected = df_freq[cell_type] > freq_threshold
        selected_spatial = df[selected]
        
        # Skip if no spots found for this cell type
        if len(selected_spatial) == 0:
            # print(f"⚠️  Skipping {cell_type}: no spots with abundance > {freq_threshold}")
            continue
        
        # Creating an object with all corrds
        all_points = df[["x", "y"]].values
        
        # Creating an oblect with only the selected points
        selected_points = selected_spatial[["x", "y"]].values
        
        # Calculate a matrix of distance between selected and all points
        dist_matrix = cdist(selected_points, all_points)
        
        # Find lower global distance and calculating the threshold distance
        # Filter out zero distances and check if there are any non-zero distances
        non_zero_distances = dist_matrix[dist_matrix > 0]
        if len(non_zero_distances) == 0:
            # print(f"⚠️  Skipping {cell_type}: no non-zero distances found")
            continue
        
        min_distance = np.min(non_zero_distances)
        threshold_distance = min_distance * distance_factor
        
        # Creating a list of neighbors
        neighbors_list = []
        
        for i, barcode in enumerate(selected_spatial.index):
            
            distances = dist_matrix[i]
            
            neighbors_idx = np.where(
                (distances < threshold_distance) &
                (distances > 0)
            )[0]
            
            x_i, y_i = selected_spatial.loc[barcode, ["x", "y"]]
            
            for j in neighbors_idx:
                
                neigh_barcode = df.index[j]
                x_j, y_j = df.iloc[j][["x", "y"]]
                
                neighbors_list.append([
                    barcode,
                    x_i,
                    y_i,
                    neigh_barcode,
                    x_j,
                    y_j,
                    distances[j]
                ])
        
        neighbors_df = pd.DataFrame(
            neighbors_list,
            columns=[
                "barcode",
                "x",
                "y",
                "neighbor_barcode",
                "x_neighbor",
                "y_neighbor",
                "distance"
            ]
        )

        # Creating result dict
        n_spots = len(neighbors_df.barcode.unique())
        # print(f"✓ {n_spots} spots found with more than {freq_threshold} for {cell_type}")
        result.update([(cell_type, n_spots)])

        # appending to a big dict
        neighbors_df["cell_type"] = cell_type
        all_neighbors_dfs.append(neighbors_df)
    
    adata.uns["deconv_result"] = result.copy()

    return pd.concat(all_neighbors_dfs, ignore_index=True)

# Calcula abundancia media de tipos celulares dos vizinhos por amostra
def get_neighbors_celltype_composition_by_sample(adata: AnnData,
                                                 sample_id: str = 'sample'):
    """
    Calcula a composição média dos vizinhos de cada tipo celular,
    armazenando um dicionário por amostra em adata.uns["neighbors_composition"].
    
    Saída: 
        adata.uns["neighbors_composition"] = dict[sample] -> DataFrame (linhas=cell_types, colunas=abundância média)
    """
    
    abundance_matrix = adata.obsm['q05_cell_abundance_w_sf']
    spatools_deconv_df = adata.uns["deconv_spatools"]

    # identificar amostras e tipos celulares
    samples = spatools_deconv_df['sample'].unique() if 'sample' in spatools_deconv_df.columns else [None]
    cell_types = spatools_deconv_df['cell_type'].unique()
    
    # dicionário final
    composition_by_sample = {}

    for sample in samples:
        # DataFrame temporário: linhas = tipos celulares, colunas = abundâncias médias dos vizinhos
        temp_df = pd.DataFrame(index=cell_types, columns=abundance_matrix.columns, dtype=float)

        for cell_type in cell_types:
            # subset dos vizinhos do tipo celular na amostra
            if sample is None:
                subset_df = spatools_deconv_df[spatools_deconv_df['cell_type'] == cell_type]
            else:
                # Filtro da amostra e tipo celular relevante
                subset_df = spatools_deconv_df[
                    (spatools_deconv_df['cell_type'] == cell_type) & 
                    (spatools_deconv_df[sample_id] == sample)
                ]

            # converte barcodes para indices numericos
            neighbors_of_type = subset_df['neighbor_barcode'].unique()
            neighbor_indices = [adata.obs_names.get_loc(barcode) 
                                for barcode in neighbors_of_type if barcode in adata.obs_names]

            # Se existirem vizinhos, faz o calculo da media das abundancias
            if len(neighbor_indices) > 0:
                neighbors_abundance = abundance_matrix.iloc[neighbor_indices]
                mean_composition = neighbors_abundance.mean(axis=0)
            else:
                mean_composition = pd.Series(0.0, index=abundance_matrix.columns)

            # salvar no DataFrame temporário
            temp_df.loc[cell_type] = mean_composition

        # salvar o DataFrame da amostra no dicionário
        composition_by_sample[sample if sample is not None else 'ALL'] = temp_df

    # armazenar no AnnData
    adata.uns["neighbors_composition"] = composition_by_sample

    return adata

# rodar measure_neighbors_filtered mas levando em consideracao que podem existir varias amostras
def deconv_correlate_distances(adata: AnnData, 
                        multi_sample: bool = True, 
                        sample_key: str = "sample",
                        freq_threshold : int = 0.05):
    
    # verifying if the analysis has already being done
    spatools_check(adata, deconv=True)

    if multi_sample:
        merged_df = []
        for i in adata.obs[sample_key].unique():
            subset = adata[adata.obs[sample_key] == i].copy()
            all_neighbors_dfs = measure_neighbors_filtered(adata=subset, freq_threshold=freq_threshold)
            all_neighbors_dfs[sample_key] = i  # add batch column
            merged_df.append(all_neighbors_dfs)

        # Concatenating individual DataFrames before storing
        adata.uns["deconv_spatools"] = pd.concat(merged_df, ignore_index=True)
        adata = get_neighbors_celltype_composition_by_sample(adata=adata)

    else:
        all_neighbors_dfs = measure_neighbors_filtered(adata=adata, freq_threshold=freq_threshold)
        adata.uns["deconv_spatools"] = all_neighbors_dfs

    return adata

bdata = deconv_correlate_distances(adata = bdata)
print(bdata.uns["neighbors_composition"])


module://matplotlib_inline.backend_inline
{'P1_GOR_S2':                        B Mem   B Naive  B Plasma  DC1_CLEC9A  DC2_AREG  \
B Mem               0.015628  0.033739  0.077764    0.006439  0.003668   
B Naive             0.006197  0.033774  0.029140    0.004869  0.002528   
B Plasma            0.007909  0.019596  0.041690    0.006665  0.003913   
DC1_CLEC9A          0.008313  0.022057  0.024803    0.012207  0.004067   
DC4_FCGR3A          0.009251  0.030828  0.038247    0.010282  0.003763   
Endothelial         0.006391  0.024913  0.031355    0.005725  0.003157   
Fibro_Inflammatory  0.007391  0.016988  0.041366    0.005455  0.003884   
Fibro_Matrix        0.007287  0.017704  0.043007    0.004694  0.003764   
Fibro_PreMatrix     0.007123  0.018303  0.038934    0.005489  0.003771   
Mac_AgPres          0.014968  0.007010  0.053492    0.011925  0.005716   
Mac_Hypo            0.008909  0.021081  0.039674    0.005161  0.003053   
Mac_LA              0.009050  0.018540  0.039679    0.00

In [4]:
bdata.uns["neighbors_composition"].keys()

dict_keys(['P1_GOR_S2', 'P2_GOR_S2', 'P3_GOR_S2', 'P4_GOR_S1', 'P5_GOR_S1', 'P6_GOR_S1', 'P7_GOR_S1', 'P8_GOR_S1', 'P9_GOR_S1', 'P10_PAR_S2', 'P11_PAR_S2', 'P12_POR_S2', 'P13_POR_S2', 'P14_POR_S2', 'P15_POR_S1', 'P16_POR_S1', 'P17_POR_S1', 'P18_POR_S1', 'P19_POR_S1'])

In [5]:
bdata.uns["neighbors_composition"]["P1_GOR_S2"]

,B Mem,B Naive,B Plasma,DC1_CLEC9A,DC2_AREG,DC2_CD207,DC2_FCER1A,DC3_CD14,DC4_FCGR3A,Endothelial,...,SmoothMuscle,TCD4_em,TCD4_ex,TCD4_naive,TCD4_reg,TCD8_em,TCD8_ex,TCD8_naive,ovary_tumor,pDC
B Mem,0.015628,0.033739,0.077764,0.006439,0.003668,0.001193,0.003361,0.001208,0.003440,0.017166,...,0.098073,0.005156,0.004865,0.146922,0.015708,0.016524,0.011352,0.010192,0.060215,0.012164
B Naive,0.006197,0.033774,0.029140,0.004869,0.002528,0.003059,0.002594,0.001885,0.010669,0.023658,...,0.057561,0.016027,0.021784,0.149735,0.021951,0.010273,0.014829,0.012429,0.329235,0.025448
B Plasma,0.007909,0.019596,0.041690,0.006665,0.003913,0.003373,0.003796,0.002084,0.007110,0.019716,...,0.077313,0.010036,0.013197,0.207689,0.018193,0.011498,0.012559,0.010809,0.211641,0.021248
DC1_CLEC9A,0.008313,0.022057,0.024803,0.012207,0.004067,0.005082,0.005361,0.003147,0.014423,0.018495,...,0.043743,0.015590,0.021915,0.190838,0.033540,0.018539,0.026810,0.017746,0.226458,0.035437
DC4_FCGR3A,0.009251,0.030828,0.038247,0.010282,0.003763,0.003720,0.005545,0.003009,0.016232,0.020180,...,0.038953,0.015411,0.023490,0.171606,0.031211,0.016972,0.023787,0.017498,0.220354,0.036900
Endothelial,0.006391,0.024913,0.031355,0.005725,0.003157,0.003394,0.003117,0.001945,0.008571,0.027347,...,0.067954,0.013027,0.017528,0.185393,0.019606,0.010575,0.013154,0.011432,0.273779,0.023713
Fibro_Inflammatory,0.007391,0.016988,0.041366,0.005455,0.003884,0.003187,0.003250,0.001790,0.005640,0.020173,...,0.094376,0.008645,0.010954,0.207215,0.013381,0.008896,0.009176,0.008265,0.213281,0.015422
Fibro_Matrix,0.007287,0.017704,0.043007,0.004694,0.003764,0.003314,0.002966,0.001741,0.005265,0.019697,...,0.101080,0.008159,0.010437,0.211761,0.011751,0.007494,0.008083,0.007212,0.217384,0.014510
Fibro_PreMatrix,0.007123,0.018303,0.038934,0.005489,0.003771,0.003261,0.003242,0.001824,0.006238,0.020869,...,0.090101,0.009547,0.012244,0.201688,0.014403,0.009063,0.009834,0.008782,0.227617,0.016650
Mac_AgPres,0.014968,0.007010,0.053492,0.011925,0.005716,0.001292,0.006084,0.001556,0.006719,0.011704,...,0.058748,0.006065,0.004253,0.167810,0.018099,0.019994,0.013076,0.013028,0.060693,0.015518


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
import seaborn as sns
import numpy as np

mask = bdata.uns["neighbors_composition"].index.get_level_values(1).str.split("_").str[1] == "GOR"
df_filtered = neighbors_composition_df[mask]
df_filtered

# plot dos resultados filtrados por resposta ao tratamento
def plot_neighbors_composition(composition_matrix: pd.DataFrame,
                               show=True, 
                               title: str = "Neighbor Cell Type Composition",
                               mask_upper=True,
                               return_object=False):
    """
    Plot the neighbors composition matrix as a heatmap.
    
    Parameters:
    -----------
    composition_matrix : pd.DataFrame
        DataFrame com tipos celulares como index e colunas
    show : bool
        Se deve mostrar o plot
    title : str
        Título do plot
    mask_upper : bool
        Se deve mascarar o triângulo superior (remove duplicatas simétricas)
    return_object : bool
        Se deve retornar o objeto do plot
    """
    
    corr_matrix = composition_matrix.copy()
    
    # --- Create mask if needed ---
    mask = None
    if mask_upper:
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    
    # --- Create custom colormap ---
    vmax = corr_matrix.values.max()
    vmin = corr_matrix.values.min()
    
    # Handle case where all values are the same
    if vmin == vmax:
        vmin = vmax - 1
    
    norm_range = vmax - vmin
    zero_pos = max(0.01, min(0.99, (0 - vmin) / norm_range if norm_range != 0 else 0.5))
    
    colors = [(0.0, '#0000FF'), (zero_pos, '#FFFFFF'), (1.0, '#FF0000')]
    cmap = LinearSegmentedColormap.from_list('custom_bwr', colors)

    # --- Plot with seaborn ---
    plt.figure(figsize=(18, 12))
    ax = sns.heatmap(corr_matrix, 
                mask=mask,
                cmap=cmap, 
                annot=True, 
                fmt='.2f',
                cbar_kws={'label': 'Mean Abundance'},
                vmin=vmin, 
                vmax=vmax,
                linewidths=0.5,
                linecolor='lightgray')

    # --- Axis and title ---
    ax.tick_params(axis="x", labelsize=11)
    ax.tick_params(axis="y", labelsize=11)

    plt.title(title, fontsize=20, pad=20)
    ax.set_xlabel('Cell Types in Neighbors', fontsize=16)
    ax.set_ylabel('Query Cell Types', fontsize=16)

    for label in ax.get_xticklabels():
        x, y = label.get_position()
        label.set_x(x + 0.5)  # adiciona metade do valor do tick

    plt.xticks(rotation=90, ha='right')
    plt.yticks(rotation=0)

    if show:
        plt.tight_layout()
        plt.show()

    if return_object:
        return corr_matrix

# Plotar composição de vizinhos (mantém apenas triângulo inferior)
plot_neighbors_composition(composition_matrix=df_filtered, 
                          title="Mean Cell Type Composition of Neighbors",
                          mask_upper=True)